In [ ]:
%pip install pandas matplotlib seaborn xlrd

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import unicodedata

sns.set_theme(style="whitegrid", palette="Set2")

In [ ]:
t2012 = pd.read_csv("Tabela5-sem_emprego_2012.csv", sep=";", encoding="utf-8-sig", decimal=",")
t2026 = pd.read_csv("Tabela5-sem_emprego_2026.csv", sep=";", encoding="utf-8-sig", decimal=",")

chaves = ["Sigla", "Código", "Estado"]
comp = t2012.merge(t2026, on=chaves, how="inner", validate="one_to_one")
comp.head()

In [ ]:
homens_2012 = "Desocupados - homens (2012 T1)"
mulheres_2012 = "Desocupados - mulheres (2012 T1)"
homens_2026 = "Desocupados - homens (2026 T1)"
mulheres_2026 = "Desocupados - mulheres (2026 T1)"

for coluna in [homens_2012, mulheres_2012, homens_2026, mulheres_2026]:
    comp[coluna] = pd.to_numeric(comp[coluna].astype(str).str.replace(",", "."), errors="coerce")

comp["Variação homens (p.p.)"] = comp[homens_2026] - comp[homens_2012]
comp["Variação mulheres (p.p.)"] = comp[mulheres_2026] - comp[mulheres_2012]
comp.sort_values("Variação mulheres (p.p.)", ascending=False).head(10)

In [ ]:
xls = pd.ExcelFile("Tabela 1.1.1.xls")
print(xls.sheet_names)
bruta = pd.read_excel(xls, sheet_name=0, header=None)
print(bruta.head(15).to_string(index=False, header=False))

In [ ]:
ufs = {
    "Acre":"AC", "Alagoas":"AL", "Amapá":"AP", "Amazonas":"AM", "Bahia":"BA",
    "Ceará":"CE", "Distrito Federal":"DF", "Espírito Santo":"ES", "Goiás":"GO",
    "Maranhão":"MA", "Mato Grosso":"MT", "Mato Grosso do Sul":"MS", "Minas Gerais":"MG",
    "Pará":"PA", "Paraíba":"PB", "Paraná":"PR", "Pernambuco":"PE", "Piauí":"PI",
    "Rio de Janeiro":"RJ", "Rio Grande do Norte":"RN", "Rio Grande do Sul":"RS",
    "Rondônia":"RO", "Roraima":"RR", "Santa Catarina":"SC", "São Paulo":"SP",
    "Sergipe":"SE", "Tocantins":"TO"
}

def normalizar(texto):
    texto = str(texto).strip().casefold()
    return "".join(c for c in unicodedata.normalize("NFKD", texto) if not unicodedata.combining(c))

mapa_ufs = {normalizar(nome): sigla for nome, sigla in ufs.items()}

# Localiza automaticamente as linhas das 27 UFs na planilha do IBGE.
base = bruta.copy()
for col in base.columns:
    base[f"uf_{col}"] = base[col].map(lambda v: mapa_ufs.get(normalizar(v)))

colunas_uf = [c for c in base.columns if str(c).startswith("uf_")]
coluna_uf = max(colunas_uf, key=lambda c: base[c].notna().sum())
linhas_uf = base[base[coluna_uf].notna()].copy()
linhas_uf["Sigla"] = linhas_uf[coluna_uf]

# Colunas numéricas da Tabela 1.1.1 (horas semanais, 2022).
colunas_dados = list(bruta.columns)
for col in colunas_dados:
    linhas_uf[col] = pd.to_numeric(
        linhas_uf[col].astype(str).str.replace(",", ".", regex=False).str.replace("…", "", regex=False),
        errors="coerce"
    )

colunas_horas = [c for c in colunas_dados if linhas_uf[c].notna().sum() >= 20]
print("UFs identificadas:", linhas_uf["Sigla"].nunique())
print("Colunas numéricas candidatas (2022):", colunas_horas)
print(bruta.iloc[:12, :].to_string(index=False, header=False))

In [ ]:
# A primeira coluna numérica candidata é usada como indicador geral.
# Confira no cabeçalho impresso acima se ela corresponde ao total de horas semanais.
coluna_total = colunas_horas[0]
horas_2022 = linhas_uf[["Sigla", coluna_total]].rename(columns={coluna_total: "Horas semanais (2022)"})
horas_2022 = horas_2022.drop_duplicates(subset="Sigla")

analise = comp.merge(horas_2022, on="Sigla", how="left", validate="one_to_one")
analise.head()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
ordem = comp.sort_values("Variação mulheres (p.p.)")
sns.barplot(data=ordem, x="Variação mulheres (p.p.)", y="Sigla", ax=ax, color="steelblue")
ax.axvline(0, color="black", linewidth=1)
ax.set(title="Mudança na participação feminina na desocupação (2012–2026)", xlabel="Variação em pontos percentuais", ylabel="UF")
plt.tight_layout()
plt.show()

In [ ]:
comparacao = comp.melt(id_vars=["Sigla", "Estado"], value_vars=[mulheres_2012, mulheres_2026], var_name="Ano", value_name="Percentual")
comparacao["Ano"] = comparacao["Ano"].map({mulheres_2012: "2012 T1", mulheres_2026: "2026 T1"})
fig, ax = plt.subplots(figsize=(14, 6))
sns.barplot(data=comparacao, x="Sigla", y="Percentual", hue="Ano", ax=ax)
ax.set(title="Composição feminina da desocupação por UF", xlabel="UF", ylabel="Participação (%)")
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
sns.scatterplot(data=analise, x="Horas semanais (2022)", y="Variação mulheres (p.p.)", ax=ax)
for _, linha in analise.dropna(subset=["Horas semanais (2022)", "Variação mulheres (p.p.)"]).iterrows():
    ax.annotate(linha["Sigla"], (linha["Horas semanais (2022)"], linha["Variação mulheres (p.p.)"]), xytext=(3, 3), textcoords="offset points", fontsize=8)
ax.axhline(0, color="gray", linestyle="--")
ax.set(title="Afazeres domésticos (2022) × mudança na desocupação feminina", xlabel="Horas semanais (2022)", ylabel="Variação feminina 2012–2026 (p.p.)")
plt.tight_layout()
plt.show()

In [ ]:
print("UFs comparadas:", len(comp))
print("UFs com dados de horas:", analise["Horas semanais (2022)"].notna().sum())
print("Média da variação feminina (p.p.):", round(comp["Variação mulheres (p.p.)"].mean(), 2))
print("Correlação entre horas de 2022 e variação feminina:", analise[["Horas semanais (2022)", "Variação mulheres (p.p.)"]].corr().iloc[0, 1])
analise.to_csv("atividade_6_comparacao.csv", index=False, sep=";", decimal=",")
analise